# 02 -- CFM audit exploration

Scratch space for the foundation-model audit. Requires the `[cfm]` extra and
downloaded checkpoints -- see `docs/CHECKPOINTS.md`.

**Run the sanity gates first.** Everything below is uninterpretable without
them: a flat bias-versus-delta curve looks like robustness and is
indistinguishable from the model not reading its conditioning input at all.


In [ ]:
import numpy as np

from bkrobust.cfm import registry, conditioning, audit
from bkrobust.data import synthetic

## What is registered

Every checkpoint must carry a pinned revision. `CheckpointSpec` refuses to
register one without.


In [ ]:
registry.available()

In [ ]:
registry.get("causalpfn")

## Gate 1 -- zero-scale identity

`bias_scale = 0.0` must reproduce `mode="none"` bit for bit. If it does not, the
injection is changing the forward pass by some route other than the intended
bias, and every number below measures that bug instead.


In [ ]:
rng = np.random.default_rng(0)
# ds = synthetic.generate_dataset(cfg.graph, rng)
# conditioning.verify_zero_scale_identity(adapter, ds["data"], bk, ds["dag"].nodes)

## Gate 2 -- true knowledge helps

The true-knowledge arm must beat the unconditioned arm, or the model is not
reading the conditioning input.


In [ ]:
# audit.sanity_check_conditioning("causalpfn", data, cpdag, dag, "X", "Y", true_ate, rng)


## What the encodings look like

Worth inspecting by eye before trusting a sweep -- an encoding whose node order
does not match the data's column order silently describes a different graph.


In [ ]:
# conditioning.knowledge_to_ancestral_matrix(bk, nodes)


In [ ]:
# conditioning.knowledge_to_adjacency_bias(bk, nodes, bias_scale=1.0)


## The sweep

Read the amortized curves against the classical arm, not in isolation. A model
that tracks OLS-on-`O*` is inheriting the same structural failure; one that does
not is failing -- or succeeding -- by some other route.


In [ ]:
# results = audit.audit_all(["causalpfn", "causalfm", "dopfn"], ...)
# audit.results_to_frame(results)
